# Projeto 1 - Cena de Pesca 3D

In [49]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math

In [50]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)
window = glfw.create_window(700, 700, "Cena de Pesca", None, None)
glfw.make_context_current(window)

In [51]:
vertex_code = """
        attribute vec3 position;
        uniform mat4 mat_transformation;
        void main(){
            gl_Position = mat_transformation * vec4(position,1.0);
        }
        """

fragment_code = """
        uniform vec4 color;
        void main(){
            gl_FragColor = color;
        }
        """

In [52]:
# Configuração dos shaders
program  = glCreateProgram()
vertex   = glCreateShader(GL_VERTEX_SHADER)
fragment = glCreateShader(GL_FRAGMENT_SHADER)

glShaderSource(vertex, vertex_code)
glShaderSource(fragment, fragment_code)

glCompileShader(vertex)
if not glGetShaderiv(vertex, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(vertex).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Vertex Shader")

glCompileShader(fragment)
if not glGetShaderiv(fragment, GL_COMPILE_STATUS):
    error = glGetShaderInfoLog(fragment).decode()
    print(error)
    raise RuntimeError("Erro de compilacao do Fragment Shader")

glAttachShader(program, vertex)
glAttachShader(program, fragment)

glLinkProgram(program)
if not glGetProgramiv(program, GL_LINK_STATUS):
    print(glGetProgramInfoLog(program))
    raise RuntimeError('Linking error')

glUseProgram(program)

In [ ]:
# Colocando a camera no ponto (3, 4, 3) e olhando para a origem, com FOV de 30 graus
view = glm.lookAt(glm.vec3(3, 4, 3), glm.vec3(0, 0, 0), glm.vec3(0, 1, 0))
projection = glm.perspective(glm.radians(30.0), 1.0, 0.1, 100.0)
mat_vp = projection * view

In [54]:
def gerar_circulo(raio, n_segmentos, y=0.0):
    """Gera triangulos de um disco no plano xz, centrado na origem."""
    vertices = []
    for i in range(n_segmentos):
        ang1 = 2.0 * math.pi * i / n_segmentos
        ang2 = 2.0 * math.pi * (i + 1) / n_segmentos
        vertices.append((0.0, y, 0.0))
        vertices.append((raio * math.cos(ang1), y, raio * math.sin(ang1)))
        vertices.append((raio * math.cos(ang2), y, raio * math.sin(ang2)))
    return vertices

def gerar_cilindro(raio, altura, n_segmentos, y_base=0.0):
    """Gera triangulos de um cilindro no eixo Y."""
    vertices = []
    y_top = y_base + altura
    for i in range(n_segmentos):
        ang1 = 2.0 * math.pi * i / n_segmentos
        ang2 = 2.0 * math.pi * (i + 1) / n_segmentos
        x1, z1 = raio * math.cos(ang1), raio * math.sin(ang1)
        x2, z2 = raio * math.cos(ang2), raio * math.sin(ang2)
        # Face lateral: 2 triangulos
        vertices.append((x1, y_base, z1))
        vertices.append((x2, y_base, z2))
        vertices.append((x1, y_top, z1))
        vertices.append((x2, y_base, z2))
        vertices.append((x2, y_top, z2))
        vertices.append((x1, y_top, z1))
    # Tampa superior
    vertices += gerar_circulo(raio, n_segmentos, y_top)
    # Tampa inferior
    vertices += gerar_circulo(raio, n_segmentos, y_base)
    return vertices

def gerar_esfera(raio, n_lat, n_lon, cy=0.0):
    """Gera triangulos de uma esfera centrada em (0, cy, 0)."""
    vertices = []
    for i in range(n_lat):
        lat1 = math.pi * i / n_lat - math.pi / 2
        lat2 = math.pi * (i + 1) / n_lat - math.pi / 2
        for j in range(n_lon):
            lon1 = 2.0 * math.pi * j / n_lon
            lon2 = 2.0 * math.pi * (j + 1) / n_lon

            x1 = raio * math.cos(lat1) * math.cos(lon1)
            y1 = raio * math.sin(lat1) + cy
            z1 = raio * math.cos(lat1) * math.sin(lon1)

            x2 = raio * math.cos(lat2) * math.cos(lon1)
            y2 = raio * math.sin(lat2) + cy
            z2 = raio * math.cos(lat2) * math.sin(lon1)

            x3 = raio * math.cos(lat2) * math.cos(lon2)
            y3 = raio * math.sin(lat2) + cy
            z3 = raio * math.cos(lat2) * math.sin(lon2)

            x4 = raio * math.cos(lat1) * math.cos(lon2)
            y4 = raio * math.sin(lat1) + cy
            z4 = raio * math.cos(lat1) * math.sin(lon2)

            vertices.append((x1, y1, z1))
            vertices.append((x2, y2, z2))
            vertices.append((x3, y3, z3))
            vertices.append((x1, y1, z1))
            vertices.append((x3, y3, z3))
            vertices.append((x4, y4, z4))
    return vertices

In [55]:
def criar_lago():
    """Disco azul no plano xz, raio 2.0"""
    return gerar_circulo(2.0, 32, y=0.0)

def criar_barco():
    """Casco de barco - prisma trapezoidal afinando nas pontas."""
    v = []
    comp = 0.5
    larg = 0.1
    alt = 0.15
    ponta = 0.08
    y_base = 0.0
    y_top = alt

    b0 = (-larg, y_base, -comp*0.8)
    b1 = ( larg, y_base, -comp*0.8)
    b2 = ( larg*0.3, y_base,  comp*0.6)
    b3 = (-larg*0.3, y_base,  comp*0.6)

    t0 = (-larg*1.3, y_top, -comp*1.15)
    t1 = ( larg*1.3, y_top, -comp*1.15)
    t2 = ( ponta,    y_top,  comp*1.1)
    t3 = (-ponta,    y_top,  comp*1.1)

    v += [b0, b1, b2,  b0, b2, b3]
    v += [b0, b3, t3,  b0, t3, t0]
    v += [b1, b2, t2,  b1, t2, t1]
    v += [b0, b1, t1,  b0, t1, t0]
    v += [b3, b2, t2,  b3, t2, t3]

    return v

def criar_pescador():
    """Boneco: esfera (cabeca) + cilindros (corpo, bracos, pernas)."""
    vertices_cabeca = gerar_esfera(0.04, 8, 8, cy=0.32)

    vertices_corpo = []
    # Corpo
    vertices_corpo += gerar_cilindro(0.035, 0.15, 8, y_base=0.15)

    # Braco esquerdo
    braco_esq = gerar_cilindro(0.015, 0.12, 6, y_base=0.15)
    braco_esq = [(x - 0.05, y, z) for x, y, z in braco_esq]
    vertices_corpo += braco_esq

    # Braco direito
    braco_dir = gerar_cilindro(0.015, 0.12, 6, y_base=0.15)
    braco_dir = [(x + 0.05, y, z) for x, y, z in braco_dir]
    vertices_corpo += braco_dir

    # Perna esquerda
    perna_esq = gerar_cilindro(0.02, 0.12, 6, y_base=0.03)
    perna_esq = [(x - 0.025, y, z) for x, y, z in perna_esq]
    vertices_corpo += perna_esq

    # Perna direita
    perna_dir = gerar_cilindro(0.02, 0.12, 6, y_base=0.03)
    perna_dir = [(x + 0.025, y, z) for x, y, z in perna_dir]
    vertices_corpo += perna_dir

    return vertices_cabeca, vertices_corpo

def criar_vara():
    """Cilindro fino longo inclinado."""
    return gerar_cilindro(0.005, 0.4, 6, y_base=0.25)

def criar_linha_vara():
    """Linha da origem e descendo verticalmente (GL_LINES)."""
    return [(0.0, 0.0, 0.0), (0.0, -0.5, 0.0)]

def criar_balde():
    """Tronco de cone (balde)."""
    vertices = []
    n_seg = 8
    raio_base = 0.03
    raio_topo = 0.04
    altura = 0.06
    y_base = 0.1
    y_top = y_base + altura
    for i in range(n_seg):
        ang1 = 2.0 * math.pi * i / n_seg
        ang2 = 2.0 * math.pi * (i + 1) / n_seg
        x1b = raio_base * math.cos(ang1)
        z1b = raio_base * math.sin(ang1)
        x2b = raio_base * math.cos(ang2)
        z2b = raio_base * math.sin(ang2)
        x1t = raio_topo * math.cos(ang1)
        z1t = raio_topo * math.sin(ang1)
        x2t = raio_topo * math.cos(ang2)
        z2t = raio_topo * math.sin(ang2)
        vertices.append((x1b, y_base, z1b))
        vertices.append((x2b, y_base, z2b))
        vertices.append((x1t, y_top, z1t))
        vertices.append((x2b, y_base, z2b))
        vertices.append((x2t, y_top, z2t))
        vertices.append((x1t, y_top, z1t))
    vertices += gerar_circulo(raio_base, n_seg, y_base)
    return vertices

def criar_peixe():
    """Peixe: elipse (corpo) + triangulo (cauda) + triangulo (barbatana)."""
    vertices = []
    n_seg = 16
    raio_x = 0.08
    raio_z = 0.03
    y = 0.001
    for i in range(n_seg):
        ang1 = 2.0 * math.pi * i / n_seg
        ang2 = 2.0 * math.pi * (i + 1) / n_seg
        vertices.append((0.0, y, 0.0))
        vertices.append((raio_x * math.cos(ang1), y, raio_z * math.sin(ang1)))
        vertices.append((raio_x * math.cos(ang2), y, raio_z * math.sin(ang2)))
    vertices.append((-raio_x, y, 0.0))
    vertices.append((-raio_x - 0.04, y, 0.025))
    vertices.append((-raio_x - 0.04, y, -0.025))
    vertices.append((0.0, y, 0.0))
    vertices.append((-0.02, y + 0.03, 0.0))
    vertices.append((0.02, y, 0.0))
    return vertices

In [ ]:
# Criando os objetos da cena
v_lago = criar_lago()
v_barco = criar_barco()
v_pescador_cabeca, v_pescador_corpo = criar_pescador()
v_vara = criar_vara()
v_linha = criar_linha_vara()
v_balde = criar_balde()
v_peixe = criar_peixe()

todos = v_lago + v_barco + v_pescador_cabeca + v_pescador_corpo + v_vara + v_balde + v_peixe + v_linha

offset = 0
objetos = {}

# Registrando os objetos com seus offsets, contagem de vertices, cor e modo de desenho
# Deixar mais organizado para desenhar
def registrar(nome, verts, cor, modo=GL_TRIANGLES):
    global offset
    objetos[nome] = {'offset': offset, 'count': len(verts), 'cor': cor, 'modo': modo}
    offset += len(verts)

registrar('lago',            v_lago,             (0.0, 0.4, 0.8, 1.0))
registrar('barco',           v_barco,            (0.55, 0.27, 0.07, 1.0))
registrar('pescador_cabeca', v_pescador_cabeca,  (0.87, 0.72, 0.53, 1.0))
registrar('pescador_corpo',  v_pescador_corpo,   (0.8, 0.1, 0.1, 1.0))
registrar('vara',            v_vara,             (0.6, 0.5, 0.2, 1.0))
registrar('balde',           v_balde,            (0.5, 0.5, 0.5, 1.0))
registrar('peixe',           v_peixe,            (0.25, 0.25, 0.25, 1.0))
registrar('linha',           v_linha,            (0.1, 0.1, 0.1, 1.0), GL_LINES)

vertices = np.zeros(len(todos), [("position", np.float32, 3)])
vertices['position'] = todos

print(f"Total de vertices: {len(todos)}")
for nome, obj in objetos.items():
    print(f"  {nome}: offset={obj['offset']}, count={obj['count']}")

Total de vertices: 1094
  lago: offset=0, count=96
  barco: offset=96, count=30
  pescador_cabeca: offset=126, count=384
  pescador_corpo: offset=510, count=384
  vara: offset=894, count=72
  balde: offset=966, count=72
  peixe: offset=1038, count=54
  linha: offset=1092, count=2


In [57]:
buffer_VBO = glGenBuffers(1)
glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO)
glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)

stride = vertices.strides[0]
loc_position = glGetAttribLocation(program, "position")
glEnableVertexAttribArray(loc_position)
glVertexAttribPointer(loc_position, 3, GL_FLOAT, False, stride, None)

loc_color = glGetUniformLocation(program, "color")
loc_mat = glGetUniformLocation(program, "mat_transformation")

In [58]:
# Estado do barco
barco_x = 0.0
barco_z = 0.0
barco_angulo = 0.0
velocidade_barco = 0.05
velocidade_rotacao = 0.05

# Oscilação do barco
barco_onda_x = 0.0
barco_onda_z = 0.0

# Estado do peixe
peixe_escala = 1.0
peixe_x = 0.5
peixe_z = 0.5
velocidade_exp_peixe = 1
peixe_angulo = 0.0

# Wireframe toggle
wireframe = False

def key_event(window, key, scancode, action, mods):
    global barco_x, barco_z, barco_angulo, barco_onda_x, barco_onda_z
    global peixe_escala, peixe_x, peixe_z, velocidade_exp_peixe, peixe_angulo
    global wireframe

    if action != glfw.PRESS and action != glfw.REPEAT:
        return

    if key == glfw.KEY_W:
        barco_x += velocidade_barco * math.sin(barco_angulo)
        barco_z += velocidade_barco * math.cos(barco_angulo)
    if key == glfw.KEY_S:
        barco_x -= velocidade_barco * math.sin(barco_angulo)
        barco_z -= velocidade_barco * math.cos(barco_angulo)

    if key == glfw.KEY_A:
        barco_angulo += velocidade_rotacao
        barco_x += velocidade_barco * math.sin(barco_angulo) * 0.5
        barco_z += velocidade_barco * math.cos(barco_angulo) * 0.5
    if key == glfw.KEY_D:
        barco_angulo -= velocidade_rotacao
        barco_x += velocidade_barco * math.sin(barco_angulo) * 0.5
        barco_z += velocidade_barco * math.cos(barco_angulo) * 0.5

    if key == glfw.KEY_Z:
        peixe_escala = max(0.2, peixe_escala - 0.1)
    if key == glfw.KEY_X:
        peixe_escala = min(3.0, peixe_escala + 0.1)

    if key == glfw.KEY_P:
        wireframe = not wireframe

glfw.set_key_callback(window, key_event)

In [59]:
def mat_to_numpy(m):
    """Converte glm.mat4 para numpy array 16 floats."""
    return np.array(m, dtype=np.float32).flatten()

In [60]:
glfw.show_window(window)

In [61]:
glEnable(GL_DEPTH_TEST)

while not glfw.window_should_close(window):
    glfw.poll_events()
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(0.2, 0.4, 0.15, 1.0)

    if wireframe:
        glPolygonMode(GL_FRONT_AND_BACK, GL_LINE)
    else:
        glPolygonMode(GL_FRONT_AND_BACK, GL_FILL)

    # Barco
    mat_barco = glm.mat4(1.0)
    mat_barco = glm.translate(mat_barco, glm.vec3(barco_x, 0.05, barco_z))
    mat_barco = glm.rotate(mat_barco, barco_angulo, glm.vec3(0, 1, 0))
    
    # Oscilação do barco
    tempo = glfw.get_time()                                              
    barco_onda_x = math.sin(tempo * 2.0) * 0.05                                                                                                          
    barco_onda_z = math.cos(tempo * 1.5) * 0.03
    mat_barco = glm.rotate(mat_barco, barco_onda_x, glm.vec3(1, 0, 0))
    mat_barco = glm.rotate(mat_barco, barco_onda_z, glm.vec3(0, 0, 1))

    # Pescador (ligada ao barco)
    mat_pescador = mat_barco * glm.translate(glm.mat4(1.0), glm.vec3(0.0, 0.0, -0.15))
    mat_pescador = glm.rotate(mat_pescador, 1, glm.vec3(0, 1, 0))

    # Vara (ligada ao pescador, inclinada para frente)
    mat_vara = mat_pescador * glm.translate(glm.mat4(1.0), glm.vec3(0.05, 0.0, -0.15))
    mat_vara = mat_vara * glm.rotate(glm.mat4(1.0), 0.75, glm.vec3(1, 0, 0))

    # Balde (ligada ao barco)
    mat_balde = mat_barco * glm.translate(glm.mat4(1.0), glm.vec3(-0.04, 0.0, -0.35))

    # Peixe (independente)
    mat_peixe = glm.mat4(1.0)
    
    # Movimento do peixe
    peixe_fator = np.power(0.001, velocidade_exp_peixe)
    if peixe_fator < 0.000005:
        velocidade_exp_peixe = 0.6
        peixe_angulo = np.random.uniform(0, 2 * np.pi)
    else:
        velocidade_exp_peixe += 0.005
    peixe_x += peixe_fator * math.sin(peixe_angulo)
    peixe_z += peixe_fator * math.cos(peixe_angulo)
    mat_peixe = glm.translate(mat_peixe, glm.vec3(peixe_x, 0.0, peixe_z))
    mat_peixe = glm.rotate(mat_peixe, peixe_angulo - np.pi/2, glm.vec3(0, 1, 0))
    mat_peixe = glm.scale(mat_peixe, glm.vec3(peixe_escala))

    # Lago (identidade)
    mat_lago = glm.mat4(1.0)

    # Linha (filha da vara)
    vara_tip = mat_vara * glm.vec4(0.0, 0.65, 0.0, 1.0)
    mat_linha = glm.translate(glm.mat4(1.0), glm.vec3(vara_tip.x, vara_tip.y, vara_tip.z))

    matrizes = {
        'lago': mat_lago,
        'barco': mat_barco,
        'pescador_cabeca': mat_pescador,
        'pescador_corpo': mat_pescador,
        'vara': mat_vara,
        'balde': mat_balde,
        'peixe': mat_peixe,
        'linha': mat_linha,
    }

    for nome, obj in objetos.items():
        mat_model = matrizes[nome]
        mat_final = mat_vp * mat_model
        glUniformMatrix4fv(loc_mat, 1, GL_TRUE, mat_to_numpy(mat_final))
        glUniform4f(loc_color, *obj['cor'])
        glDrawArrays(obj['modo'], obj['offset'], obj['count'])

    glfw.swap_buffers(window)

glfw.terminate()